# Proyecto Final Big Data — Tokio Telecom
## Análisis de bajas (churn) de clientes con **PySpark** en Google Colab

**Dataset:** `customer_churn_10k.csv` (10.000 clientes, 12 columnas)

Este notebook resuelve las **tareas 3 a 6** del proyecto usando exclusivamente **Python + Spark (PySpark)**:

- **T3** — Carga en DataFrame + análisis de churn por tipo de tarifa y por duración de contrato.
- **T4** — Análisis descriptivo (media y desviación estándar).
- **T5** — Análisis de correlación (matriz de Pearson).
- **T6** — Aprendizaje supervisado: Random Forest + Regresión Logística (baseline) para predecir la baja.


## 0. Instalación de PySpark en Colab
Colab ya trae Java. Solo instalamos la librería `pyspark`.

In [ ]:
!pip install pyspark

## 1. Crear la sesión de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (SparkSession.builder
         .appName('Tokio_Telecom_Churn')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

## 2. Subir y cargar el CSV
Ejecuta la celda y selecciona `customer_churn_10k.csv` desde tu equipo.
(Alternativa: súbelo al panel de archivos de Colab y comenta la línea de `files.upload()`.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ruta = 'customer_churn_10k.csv'

### 2.1 Cargar el conjunto de datos en un DataFrame de PySpark

In [ ]:
df = spark.read.csv(ruta, header=True, inferSchema=True)
# CustomerID debe tratarse como CATEGÓRICA y no usarse en los modelos
df = df.withColumn('CustomerID', F.col('CustomerID').cast('string'))
print('Filas:', df.count(), '| Columnas:', len(df.columns))
df.printSchema()
df.show(5, truncate=False)

### 2.2 Estructura de los datos (Nombre del campo / Tipo)
Referencia de la Tarea 1 — tipos con los que trabajaremos:

In [ ]:
import pandas as pd
tipos = pd.DataFrame({
    'Nombre del Campo': [f.name for f in df.schema.fields],
    'Tipo de dato':     [f.dataType.simpleString() for f in df.schema.fields]
})
tipos

---
## Tarea 3 — Clientes y % de churn por categoría
Calculamos, para **Subscription Type** y **Contract Length**, cuántos clientes hay y qué porcentaje ha hecho churn (`Churn_YesNo = 'Yes'`).

In [ ]:
def churn_por(col):
    return (df.groupBy(col)
              .agg(F.count('*').alias('clientes'),
                   F.sum(F.when(F.col('Churn_YesNo')=='Yes',1).otherwise(0)).alias('churn_yes'))
              .withColumn('pct_churn', F.round(100*F.col('churn_yes')/F.col('clientes'),2))
              .orderBy(F.desc('pct_churn')))

t3_sub = churn_por('Subscription Type')
t3_con = churn_por('Contract Length')
t3_sub.show(truncate=False)
t3_con.show(truncate=False)

### Gráficos de la Tarea 3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
AZUL, ROSA = '#1a1aa8', '#f96b7d'

def grafico_churn(spark_df, catcol, titulo):
    pdf = spark_df.toPandas().sort_values('pct_churn', ascending=False).reset_index(drop=True)
    fig, ax1 = plt.subplots(figsize=(7,4.3))
    x = np.arange(len(pdf))
    ax1.bar(x-0.2, pdf['clientes'], 0.4, color=AZUL, label='Nº clientes')
    ax1.set_ylabel('Nº de clientes', color=AZUL); ax1.tick_params(axis='y', labelcolor=AZUL)
    ax2 = ax1.twinx()
    ax2.bar(x+0.2, pdf['pct_churn'], 0.4, color=ROSA, label='% churn')
    ax2.set_ylabel('% churn (Yes)', color=ROSA); ax2.tick_params(axis='y', labelcolor=ROSA)
    ax2.set_ylim(0, max(105, pdf['pct_churn'].max()*1.2))
    for i,v in enumerate(pdf['pct_churn']): ax2.text(i+0.2, v+1.5, f'{v:.1f}%', ha='center', color=ROSA, fontweight='bold', fontsize=9)
    ax1.set_xticks(x); ax1.set_xticklabels(pdf[catcol]); ax1.set_title(titulo, fontweight='bold')
    fig.tight_layout(); plt.show()

grafico_churn(t3_sub, 'Subscription Type', 'Churn por tipo de tarifa')
grafico_churn(t3_con, 'Contract Length', 'Churn por duración del contrato')

> **Interpretación de negocio (T3).** Por tarifa el churn es alto y homogéneo (Basic 58,3% · Standard 56,7% · Premium 55,3%): el tipo de tarifa apenas discrimina la baja. En cambio la **duración del contrato es determinante**: el contrato **mensual presenta un 100% de churn**, frente al ~47% del trimestral y ~45% del anual. El compromiso a medio/largo plazo es la palanca de retención más potente.

---
## Tarea 4 — Análisis descriptivo (media y desviación estándar)
Solo variables **numéricas** (excluimos `CustomerID`, que es categórica).

In [ ]:
num_cols = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction']
desc = df.select([F.round(F.mean(c),3).alias(c+'_media') for c in num_cols] +
                 [F.round(F.stddev(c),3).alias(c+'_desv') for c in num_cols])
# Presentación en formato largo y legible
filas = []
for c in num_cols:
    r = df.select(F.mean(c).alias('m'), F.stddev(c).alias('s'),
                  F.min(c).alias('mn'), F.max(c).alias('mx')).first()
    filas.append((c, round(r['m'],3), round(r['s'],3), r['mn'], r['mx']))
import pandas as pd
pd.DataFrame(filas, columns=['Variable','Media','Desv. Estándar','Mín','Máx'])

> **Interpretación (T4).** Edad media 39 años (18–65). `Tenure` (antigüedad) media 31 meses con alta dispersión (±17). `Support Calls` media 3,6 y `Payment Delay` media 13 días: variables de fricción con recorrido. `Total Spend` medio 633 € (±240), muy dispersa → conviven clientes de bajo y alto valor.

---
## Tarea 5 — Análisis de correlación
Para calcular la matriz de correlación necesitamos **ensamblar** las variables numéricas en un vector (`VectorAssembler`) y aplicar `Correlation.corr` (Pearson).

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

df_num = df.select([F.col(c).cast(DoubleType()).alias(c) for c in num_cols])
vec = VectorAssembler(inputCols=num_cols, outputCol='features_corr').transform(df_num).select('features_corr')
M = Correlation.corr(vec, 'features_corr', 'pearson').head()[0].toArray()
import pandas as pd
corr_df = pd.DataFrame(M, index=num_cols, columns=num_cols).round(3)
corr_df

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(figsize=(7.5,6.2))
im = ax.imshow(M, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([c.replace(' ','\n') for c in num_cols], fontsize=8); ax.set_yticklabels(num_cols, fontsize=8)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j,i,f'{M[i,j]:.2f}', ha='center', va='center', color='white' if abs(M[i,j])>0.5 else 'black', fontsize=8)
ax.set_title('Matriz de correlación (Pearson)', fontweight='bold'); fig.colorbar(im, fraction=0.046, pad=0.04)
fig.tight_layout(); plt.show()

### Correlación de cada variable con la baja
Añadimos una columna numérica `churn01` (Yes=1) para ver qué variables se asocian más con el churn.

In [ ]:
df_c = df.withColumn('churn01', F.when(F.col('Churn_YesNo')=='Yes',1.0).otherwise(0.0))
for c in num_cols:
    print(f'{c:18s} corr con churn = {df_c.stat.corr(c, "churn01"):.3f}')

> **Interpretación (T5).** Entre las variables numéricas la correlación es muy baja (no hay multicolinealidad → todas aportan información propia a un modelo). Frente a la baja destacan: **Support Calls (+0,58)**, **Total Spend (−0,42)**, **Payment Delay (+0,32)** y **Age (+0,23)**. Lectura: más llamadas a soporte y más retrasos de pago empujan a la baja; más gasto (cliente más comprometido/valioso) la reduce.

---
## Tarea 6 — Aprendizaje supervisado (predecir la baja)

**Problema de negocio.** Anticipar qué clientes se van a dar de baja permite actuar (retención)
*antes* de perderlos, protegiendo ingresos recurrentes. Es una **clasificación binaria** con etiqueta
conocida (`Churn_YesNo`), así que aplicamos **aprendizaje supervisado**.

**Modelos comparados.**
1. **Árbol de Decisión** — interpretable, da reglas directas.
2. **Random Forest** — *ensemble* de árboles; reduce varianza y suele mejorar el AUC.
3. **Regresión Logística** — *baseline* lineal de referencia.

Los tres se afinan con **Grid Search + Validación Cruzada (5-fold)** optimizando el **AUC**.

**Procesado de categóricas.** `Gender`, `Subscription Type` y `Contract Length` → `StringIndexer` +
`OneHotEncoder`. `CustomerID` se excluye (identificador). Para la Regresión Logística escalamos con
`StandardScaler`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# Etiqueta: Yes = 1 (la baja es la CLASE POSITIVA: es el evento que queremos detectar)
data = df.withColumn('label', F.when(F.col('Churn_YesNo')=='Yes',1.0).otherwise(0.0))

numeric_cols     = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction']
categorical_cols = ['Gender','Subscription Type','Contract Length']

indexers  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in categorical_cols]
encoders  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in categorical_cols]
assembler = VectorAssembler(inputCols=numeric_cols+[c+'_ohe' for c in categorical_cols], outputCol='features')
scaler    = StandardScaler(inputCol='features', outputCol='scaledFeatures', withStd=True, withMean=False)
pre = indexers + encoders + [assembler, scaler]

### 6.1 Estrategia de validación y elección de métrica

Para entrenar **correctamente** los modelos (no con hiperparámetros por defecto):

- **Partición train/test 70/30** con semilla fija (reproducibilidad).
- **Grid Search + Validación Cruzada (5-fold):** probamos varias combinaciones de hiperparámetros y
  evaluamos cada una en 5 particiones; nos quedamos con la que maximiza el **AUC** medio. Así el
  resultado no depende de un único reparto de datos.
- **Métrica de referencia: AUC (área bajo la curva ROC), NO la *accuracy*.** En churn importa
  **ordenar** a los clientes por riesgo (se actúa sobre el top-N con presupuesto limitado) y el
  **coste asimétrico** (perder un cliente cuesta más que un incentivo). Además la accuracy se mide a un
  umbral fijo (0,5), mientras que el AUC evalúa **todos** los umbrales.
- Reportamos métricas **en train y en test** para vigilar el **sobreajuste**.

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc as sk_auc
import numpy as np, pandas as pd, matplotlib.pyplot as plt

train, test = data.randomSplit([0.7, 0.3], seed=42)
train.cache(); test.cache()
print('Train:', train.count(), '| Test:', test.count())

auc_ev = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')

def metricas(model, dfset):
    p = model.transform(dfset)
    auc = auc_ev.evaluate(p)
    f1  = MulticlassClassificationEvaluator(labelCol='label', metricName='f1').evaluate(p)
    acc = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy').evaluate(p)
    cm = {(int(r['label']),int(r['prediction'])): r['count']
          for r in p.groupBy('label','prediction').count().collect()}
    tp=cm.get((1,1),0); fn=cm.get((1,0),0); fp=cm.get((0,1),0); tn=cm.get((0,0),0)
    rec  = tp/(tp+fn) if (tp+fn) else 0.0     # recall de la clase baja
    prec = tp/(tp+fp) if (tp+fp) else 0.0     # precision de la clase baja
    return dict(auc=auc, f1=f1, acc=acc, recall=rec, precision=prec, cm=[[tn,fp],[fn,tp]])

def roc_points(model, dfset):
    pdf = (model.transform(dfset).withColumn('p1', vector_to_array('probability')[1])
             .select('label','p1').toPandas())
    fpr,tpr,_ = roc_curve(pdf['label'], pdf['p1']); return fpr, tpr, sk_auc(fpr,tpr)

def pget(model, name):
    getter='get'+name[0].upper()+name[1:]
    if hasattr(model, getter):
        v=getattr(model,getter); return v() if callable(v) else v
    return getattr(model, name)

resultados = {}

### 6.2 Árbol de Decisión (Grid Search + Cross Validation)

In [ ]:
dt = DecisionTreeClassifier(labelCol='label', featuresCol='features', seed=42)
grid_dt = (ParamGridBuilder()
           .addGrid(dt.maxDepth, [3, 5, 8, 12])
           .addGrid(dt.minInstancesPerNode, [1, 20])
           .addGrid(dt.impurity, ['gini', 'entropy']).build())
cv_dt = CrossValidator(estimator=Pipeline(stages=pre+[dt]), estimatorParamMaps=grid_dt,
                       evaluator=auc_ev, numFolds=5, parallelism=4, seed=42)
m_dt = cv_dt.fit(train); best_dt = m_dt.bestModel.stages[-1]
resultados['Árbol de Decisión'] = dict(model=m_dt.bestModel, cv_auc=float(max(m_dt.avgMetrics)),
    best_params={'maxDepth':pget(best_dt,'maxDepth'),'minInstancesPerNode':pget(best_dt,'minInstancesPerNode'),'impurity':pget(best_dt,'impurity')},
    train=metricas(m_dt.bestModel, train), test=metricas(m_dt.bestModel, test))
print('Mejores hiperparámetros:', resultados['Árbol de Decisión']['best_params'])
r = resultados['Árbol de Decisión']
print('AUC  CV=%.4f | train=%.4f | test=%.4f' % (r['cv_auc'], r['train']['auc'], r['test']['auc']))

### 6.3 Random Forest (Grid Search + Cross Validation)

In [ ]:
rf = RandomForestClassifier(labelCol='label', featuresCol='features', seed=42)
grid_rf = (ParamGridBuilder()
           .addGrid(rf.numTrees, [50, 120])
           .addGrid(rf.maxDepth, [5, 8, 12])
           .addGrid(rf.featureSubsetStrategy, ['sqrt', 'onethird']).build())
cv_rf = CrossValidator(estimator=Pipeline(stages=pre+[rf]), estimatorParamMaps=grid_rf,
                       evaluator=auc_ev, numFolds=5, parallelism=4, seed=42)
m_rf = cv_rf.fit(train); best_rf = m_rf.bestModel.stages[-1]
resultados['Random Forest'] = dict(model=m_rf.bestModel, cv_auc=float(max(m_rf.avgMetrics)),
    best_params={'numTrees':pget(best_rf,'numTrees'),'maxDepth':pget(best_rf,'maxDepth'),'featureSubsetStrategy':pget(best_rf,'featureSubsetStrategy')},
    train=metricas(m_rf.bestModel, train), test=metricas(m_rf.bestModel, test))
print('Mejores hiperparámetros:', resultados['Random Forest']['best_params'])
r = resultados['Random Forest']
print('AUC  CV=%.4f | train=%.4f | test=%.4f' % (r['cv_auc'], r['train']['auc'], r['test']['auc']))

### 6.4 Regresión Logística (baseline, con Cross Validation)

In [ ]:
lr = LogisticRegression(labelCol='label', featuresCol='scaledFeatures', maxIter=100)
grid_lr = (ParamGridBuilder()
           .addGrid(lr.regParam, [0.0, 0.01, 0.1])
           .addGrid(lr.elasticNetParam, [0.0, 0.5]).build())
cv_lr = CrossValidator(estimator=Pipeline(stages=pre+[lr]), estimatorParamMaps=grid_lr,
                       evaluator=auc_ev, numFolds=5, parallelism=4, seed=42)
m_lr = cv_lr.fit(train); best_lr = m_lr.bestModel.stages[-1]
resultados['Regresión Logística'] = dict(model=m_lr.bestModel, cv_auc=float(max(m_lr.avgMetrics)),
    best_params={'regParam':pget(best_lr,'regParam'),'elasticNetParam':pget(best_lr,'elasticNetParam')},
    train=metricas(m_lr.bestModel, train), test=metricas(m_lr.bestModel, test))
print('Mejores hiperparámetros:', resultados['Regresión Logística']['best_params'])
r = resultados['Regresión Logística']
print('AUC  CV=%.4f | train=%.4f | test=%.4f' % (r['cv_auc'], r['train']['auc'], r['test']['auc']))

### 6.5 Tabla comparativa **test–train–modelos**
Métrica de referencia: **AUC**. Se muestran train y test para detectar sobreajuste; incluimos también
recall/precision de la clase *baja* y la accuracy (secundaria).

In [ ]:
orden = ['Árbol de Decisión', 'Random Forest', 'Regresión Logística']
filas = []
for nom in orden:
    r = resultados[nom]
    filas.append([nom, r['cv_auc'], r['train']['auc'], r['test']['auc'],
                  r['test']['f1'], r['test']['recall'], r['test']['precision'], r['test']['acc']])
tabla = pd.DataFrame(filas, columns=['Modelo','AUC (CV)','AUC train','AUC test','F1 test',
                                     'Recall churn test','Precision churn test','Accuracy test'])
tabla.round(4)

### 6.6 Curvas ROC — train y test
La curva ROC enfrenta sensibilidad (verdaderos positivos) frente a 1−especificidad (falsos positivos)
en todos los umbrales. Que **train ≈ test** indica que **no hay sobreajuste**.

In [ ]:
roc = {nom: dict(tr=roc_points(resultados[nom]['model'], train),
                 te=roc_points(resultados[nom]['model'], test)) for nom in orden}
COL = {'Árbol de Decisión':'#f96b7d', 'Random Forest':'#1a1aa8', 'Regresión Logística':'#5bc4a8'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, key, tit in [(axes[0],'tr','Curvas ROC — TRAIN (70%)'), (axes[1],'te','Curvas ROC — TEST (30%)')]:
    for nom in orden:
        fpr,tpr,a = roc[nom][key]
        ax.plot(fpr, tpr, color=COL[nom], lw=2.2, label=f'{nom} (AUC={a:.3f})')
    ax.plot([0,1],[0,1],'--',color='#9aa0b4',lw=1,label='Azar (0.5)')
    ax.set_xlim(0,1); ax.set_ylim(0,1.01); ax.grid(alpha=.25)
    ax.set_xlabel('1 - especificidad (FPR)'); ax.set_ylabel('sensibilidad (TPR)')
    ax.set_title(tit, fontweight='bold'); ax.legend(loc='lower right', fontsize=8.5)
fig.suptitle('Curvas ROC-AUC por modelo — Train vs Test', fontweight='bold'); fig.tight_layout(); plt.show()

In [ ]:
x = np.arange(len(orden)); w = 0.38
tr = [resultados[n]['train']['auc'] for n in orden]; te = [resultados[n]['test']['auc'] for n in orden]
fig, ax = plt.subplots(figsize=(8,4.6))
b1=ax.bar(x-w/2, tr, w, color='#1a1aa8', label='AUC train'); b2=ax.bar(x+w/2, te, w, color='#f96b7d', label='AUC test')
ax.set_xticks(x); ax.set_xticklabels(orden); ax.set_ylim(0,1.08); ax.set_ylabel('AUC')
ax.set_title('AUC train vs test — control de sobreajuste', fontweight='bold')
for bs in (b1,b2):
    for b in bs: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}', ha='center', fontsize=8.5)
ax.legend(); fig.tight_layout(); plt.show()

### 6.7 Matrices de confusión (test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, nom in zip(axes, orden):
    M = np.array(resultados[nom]['test']['cm']); ax.imshow(M, cmap='Blues')
    ax.set_title(nom, fontweight='bold', fontsize=10)
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['No','Yes']); ax.set_yticklabels(['No','Yes'])
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
    for i in range(2):
        for j in range(2):
            ax.text(j,i,int(M[i,j]),ha='center',va='center',color='white' if M[i,j]>M.max()/2 else 'black',fontsize=12,fontweight='bold')
fig.suptitle('Matrices de confusión (test) — modelos afinados', fontweight='bold'); fig.tight_layout(); plt.show()

### 6.8 Importancia de variables (Random Forest afinado)

In [ ]:
feat_names = list(numeric_cols)
idx_stages = {s.getInputCol(): s for s in resultados['Random Forest']['model'].stages if hasattr(s,'labels')}
for c in categorical_cols:
    for lab in idx_stages[c].labels[:-1]:
        feat_names.append(f'{c}={lab}')
imp = best_rf.featureImportances.toArray()
pares = sorted(zip(feat_names, imp), key=lambda z:z[1], reverse=True)
for n,v in pares: print(f'{n:28s} {v:.4f}')
top = pares[:10][::-1]
fig, ax = plt.subplots(figsize=(8,5))
ax.barh([t[0] for t in top],[t[1] for t in top], color='#5bc4a8')
ax.set_title('Importancia de variables — Random Forest afinado', fontweight='bold'); ax.set_xlabel('Importancia relativa')
for i,t in enumerate(top): ax.text(t[1]+0.003, i, f'{t[1]:.3f}', va='center', fontsize=8)
fig.tight_layout(); plt.show()

> **Conclusiones del modelo (T6).**
>
> | Modelo | AUC (CV) | AUC train | AUC test | Recall baja (test) |
> |---|---|---|---|---|
> | Árbol de Decisión | 0.9943 | 0.9996 | 0.9982 | 0.997 |
> | **Random Forest** | 0.9994 | 1.0000 | **0.9997** | 0.990 |
> | Regresión Logística | 0.9599 | 0.9602 | 0.9583 | 0.896 |
>
> - **Mejor modelo: Random Forest** (hiperparámetros por Grid Search: numTrees=120,
>   maxDepth=12, featureSubsetStrategy='onethird'), con
>   **AUC test = 0.9997**. El Árbol de Decisión afinado
>   (maxDepth=12, impurity='entropy') queda muy cerca (0.9982) y es más interpretable.
> - **No hay sobreajuste:** el AUC en **train ≈ test** en los tres modelos (RF: 1.0000 vs 0.9997) y coincide con el de **validación cruzada**.
> - **¿Por qué un AUC tan alto?** El dataset (sintético) contiene reglas casi deterministas
>   (contrato *Mensual* → 100% de baja; *Support Calls* ≥ 6 → 100%; *Payment Delay* > 20 → 100%;
>   *Total Spend* ≤ 500 → 100%). El techo del problema es muy alto: no es un artefacto del modelo.
> - **La accuracy no es la métrica adecuada** (coste asimétrico + necesidad de *ranking*): por eso
>   optimizamos y reportamos **AUC** y la **curva ROC**.
> - Variables más determinantes (RF): **Support Calls** (0,31), **Total Spend** (0,22), **Age** (0,13)
>   y **Payment Delay** (0,12) — coherente con el análisis de correlación (T5).
>
> **Próximos pasos.** Validación temporal (out-of-time), recalibración de probabilidades y elección de
> umbral por coste, probar *Gradient-Boosted Trees*, explicabilidad con SHAP, e industrializar el
> scoring por lotes hacia el CRM con monitorización de *data drift*.

In [ ]:
spark.stop()